In [9]:
import pandas as pd
import pickle as pkl
import numpy as np
from matplotlib import pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [3]:
data = pkl.load(
    open(
        "../data/generated/r2_null_tests/f7d46a15-9498-40dc-90da-fb977ce844be_region_predictions.pkl",
        "rb",
    )
)

In [4]:
true_r2 = data["true_r2"]
null_r2 = data["null_r2"]

In [10]:
def find_significant_pairs(true_r2, null_r2, alpha=0.05):

    n_regions = true_r2.shape[0]
    n_perms = null_r2.shape[2]

    off_diag_mask = ~np.eye(n_regions, dtype=bool)

    valid_true = true_r2[off_diag_mask]
    valid_null = null_r2[off_diag_mask]

    exceedances = np.sum(valid_null >= valid_true[:, None], axis=1)
    p_values_raw = (exceedances + 1) / (n_perms + 1)

    reject, p_values_corrected, _, _ = multipletests(p_values_raw, alpha=alpha, method="fdr_bh")

    sig_matrix = np.zeros((n_regions, n_regions), dtype=bool)
    p_matrix_adj = np.ones((n_regions, n_regions))
    sig_matrix[off_diag_mask] = reject
    p_matrix_adj[off_diag_mask] = p_values_corrected

    return sig_matrix, p_matrix_adj

In [11]:
sig_matrix, p_matrix = find_significant_pairs(true_r2, null_r2)